In [239]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [240]:
df = sns.load_dataset("titanic")

In [241]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [242]:
df = df.drop([ 'deck' , 'alone' , 'adult_male' , 'alive' , 'who' , 'embark_town' ], axis=1)
df.dropna(inplace=True)
df.head()
df[ 'family' ]=df['parch']+df['sibsp']

In [243]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,family
0,0,3,male,22.0,1,0,7.2500,S,Third,1
1,1,1,female,38.0,1,0,71.2833,C,First,1
2,1,3,female,26.0,0,0,7.9250,S,Third,0
3,1,1,female,35.0,1,0,53.1000,S,First,1
4,0,3,male,35.0,0,0,8.0500,S,Third,0


In [244]:
x=df.drop(['survived'],axis=1)
y=df['survived']
from sklearn.model_selection import train_test_split
x_train , x_test , y_train , y_test = train_test_split(x,y,test_size=0.2 , random_state=42)

In [245]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OrdinalEncoder , LabelEncoder,MinMaxScaler,OneHotEncoder
ct = ColumnTransformer(transformers=[
    ('sex', OneHotEncoder(drop='first'), ['sex']),
    ('class', OrdinalEncoder(categories=[['Third', 'Second', 'First']]), ['class']),
    ('fare', StandardScaler(), ['fare']),
    ('age', MinMaxScaler(), ['age']),
    ('embarked', OneHotEncoder(drop='first'), ['embarked']),
    ('num', StandardScaler(), ['sibsp','parch','family'])
], remainder='passthrough')

In [246]:
from sklearn.pipeline import make_pipeline

In [247]:
from sklearn.metrics import accuracy_score , recall_score , precision_score , f1_score,confusion_matrix 
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [248]:
mdl=make_pipeline(ct,RandomForestClassifier(n_estimators=100, max_depth=5 , max_features=3))
# mdl=make_pipeline(ct,LogisticRegression())
# mdl=LogisticRegression()
mdl.fit(x_train,y_train)
y_pred=mdl.predict(x_test)

In [249]:
print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(f1_score(y_test,y_pred))
confusion_matrix(y_test,y_pred)

0.7832167832167832
0.8478260869565217
0.6190476190476191
0.7155963302752294


array([[73,  7],
       [24, 39]], dtype=int64)

In [250]:
n_estimators = [20,60,100,120]
max_features=[0.2,0.6,1.0]

max_depth=[2,8,None]

max_samples=[0.5,0.75,1.0]

 

In [251]:
param_grid={
    'n_estimators':n_estimators,
    'max_features':max_features,
    'max_depth':max_depth,
    'max_samples':max_samples
}

In [252]:
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(estimator=RandomForestClassifier(),
                       param_grid=param_grid,
                       cv=5,
                       verbose=2,
                       n_jobs=-1)

In [255]:
mdl=make_pipeline(ct,RandomForestClassifier(max_depth= 8, max_features= 0.6, max_samples= 0.5, n_estimators= 100))
mdl.fit(x_train,y_train)
y_pred=mdl.predict(x_test)

print(accuracy_score(y_test,y_pred))
print(precision_score(y_test,y_pred))
print(recall_score(y_test,y_pred))
print(f1_score(y_test,y_pred))
confusion_matrix(y_test,y_pred)

0.7972027972027972
0.84
0.6666666666666666
0.7433628318584071


array([[72,  8],
       [21, 42]], dtype=int64)

AttributeError: 'GridSearchCV' object has no attribute 'best_params_'